# Amsterdam Co-Accessibility with CBS Population Data

This notebook estimates **co-accessibility** for Amsterdam from the POI perspective using CBS 100x100m population data.

It answers: **how many people from different population groups can access the same POI?**


In [ ]:
import accessx as acx
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import pandas as pd
from pathlib import Path


## Setup


In [ ]:
epsg_ams = 28992
resolution = 9
buffer_m = 1000
max_cost = 15
cost_attr = "avg_time"

coacc_path = Path("../data/case_studies/coaccessibility/amsterdam")
area_path = coacc_path / "area"
population_path = coacc_path / "population"
street_network_path = coacc_path / "street_network"
pois_path = coacc_path / "pois"
results_path = coacc_path / "results"

for folder in [coacc_path, area_path, population_path, street_network_path, pois_path, results_path]:
    folder.mkdir(parents=True, exist_ok=True)

cbs_path = Path("../data/nl_cbs/cbs_vk100_2020_vol.gpkg")

demographic_columns = {
    "population": "aantal_inwoners",
    "men": "aantal_mannen",
    "women": "aantal_vrouwen",
    "age_0_15": "aantal_inwoners_0_tot_15_jaar",
    "age_65_plus": "aantal_inwoners_65_jaar_en_ouder",
}

tags_library = {
    "amenity_pharmacy": {"amenity": ["pharmacy"]},
    "amenity_restaurant": {"amenity": ["restaurant"]},
    "leisure_playground": {"leisure": ["playground"]},
}

poi_categories = list(tags_library.keys())


## Load Area of Interest


In [ ]:
gdf_ams = ox.geocode_to_gdf("Amsterdam, Netherlands")
acx.save_gdf(gdf_ams, area_path / "aoi_amsterdam.geojson")
# gdf_ams = acx.read_gdf(area_path / "aoi_amsterdam.geojson")


## Make Hex Grid


In [ ]:
hex_ams = acx.make_hex_grid(gdf_ams, resolution=resolution, clip=True)
acx.save_gdf(hex_ams, area_path / "hexes_amsterdam.geojson")
# hex_ams = acx.read_gdf(area_path / "hexes_amsterdam.geojson")


## Load and Prepare CBS Population Grid


In [ ]:
ams_bbox = tuple(gdf_ams.to_crs(epsg_ams).total_bounds)
cbs_ams = gpd.read_file(cbs_path, bbox=ams_bbox)
cbs_ams = cbs_ams.to_crs(epsg_ams)
cbs_ams = gpd.overlay(
    cbs_ams[["geometry"] + list(demographic_columns.values())],
    gdf_ams.to_crs(epsg_ams)[["geometry"]],
    how="intersection",
    keep_geom_type=False,
)

for source_col in demographic_columns.values():
    cbs_ams[source_col] = pd.to_numeric(cbs_ams[source_col], errors="coerce")
    cbs_ams[source_col] = cbs_ams[source_col].where(cbs_ams[source_col] >= 0, 0.0)

cbs_ams = cbs_ams.rename(columns={source: target for target, source in demographic_columns.items()})
acx.save_gdf(cbs_ams, population_path / "cbs_amsterdam_clipped.geojson")
# cbs_ams = acx.read_gdf(population_path / "cbs_amsterdam_clipped.geojson")

cbs_ams[["population", "men", "women", "age_0_15", "age_65_plus"]].describe()


## Map CBS Population Groups to Hexes


In [ ]:
hex_pop_ams = acx.map_population_grid_to_hexes(
    hex_ams,
    cbs_ams,
    metric_crs=epsg_ams,
    population_cols=["population", "men", "women", "age_0_15", "age_65_plus"],
)
acx.save_gdf(hex_pop_ams, population_path / "hexes_population_amsterdam_cbs.geojson")
# hex_pop_ams = acx.read_gdf(population_path / "hexes_population_amsterdam_cbs.geojson")

hex_pop_ams[["population", "men", "women", "age_0_15", "age_65_plus"]].sum()


## Build Street Network


In [ ]:
graph_ams = acx.build_network(
    AOI=gdf_ams,
    city_epsg=epsg_ams,
    buffer_m=buffer_m,
    network_type="walk",
    simplify=False,
    retain_all=True,
)

graph_ams = acx.add_time_cost_constant_speed(
    graph_ams,
    speed_kmh=4.5,
    cost_col=cost_attr,
)

acx.save_graph(
    graph_ams,
    out_dir=street_network_path,
    base_name="amsterdam_graph_with_cost",
    save_nodes=True,
    save_edges=True,
)

# graph_ams = acx.load_graph(
#     nodes_path=street_network_path / "amsterdam_graph_with_cost_nodes_OSM.geojson",
#     edges_path=street_network_path / "amsterdam_graph_with_cost_edges_OSM.geojson",
#     crs=epsg_ams,
# )


## Collect POIs


In [ ]:
pois_ams = acx.get_pois_osm(gdf_ams, tags_library=tags_library)
acx.save_gdf(pois_ams, pois_path / "pois_amsterdam.geojson")
# pois_ams = acx.read_gdf(pois_path / "pois_amsterdam.geojson")

pois_ams["category"].value_counts()


## Compute Cumulative Co-Accessibility


In [ ]:
coacc_cumulative = acx.compute_co_accessibility(
    graph_ams,
    hex_pop_ams,
    pois_ams,
    max_cost=max_cost,
    cost_attr=cost_attr,
    population_groups=["population", "men", "women", "age_0_15", "age_65_plus"],
    poi_id_col="id",
    category_col="category",
    approach="cumulative",
)
acx.save_gdf(coacc_cumulative, results_path / "coaccessibility_cumulative_amsterdam.geojson")
# coacc_cumulative = acx.read_gdf(results_path / "coaccessibility_cumulative_amsterdam.geojson")

coacc_cumulative.head()


## Compute Hansen Co-Accessibility


In [ ]:
coacc_hansen = acx.compute_co_accessibility(
    graph_ams,
    hex_pop_ams,
    pois_ams,
    max_cost=max_cost,
    cost_attr=cost_attr,
    population_groups=["population", "men", "women", "age_0_15", "age_65_plus"],
    poi_id_col="id",
    category_col="category",
    approach="hansen",
    beta=0.15,
)
acx.save_gdf(coacc_hansen, results_path / "coaccessibility_hansen_amsterdam.geojson")
# coacc_hansen = acx.read_gdf(results_path / "coaccessibility_hansen_amsterdam.geojson")

coacc_hansen.head()


## Summarize Co-Accessibility by POI Category


In [ ]:
summary_cumulative = (
    coacc_cumulative.groupby("category")[["coacc_population", "coacc_men", "coacc_women", "coacc_age_0_15", "coacc_age_65_plus"]]
    .agg(["mean", "median", "max"])
)
summary_cumulative


In [ ]:
summary_hansen = (
    coacc_hansen.groupby("category")[["coacc_population", "coacc_men", "coacc_women", "coacc_age_0_15", "coacc_age_65_plus"]]
    .agg(["mean", "median", "max"])
)
summary_hansen


## Inspect the Top POIs by Reachable Population


In [ ]:
top_cumulative = (
    coacc_cumulative[["id", "name", "category", "coacc_population", "coacc_age_0_15", "coacc_age_65_plus"]]
    .sort_values("coacc_population", ascending=False)
    .head(15)
)
top_cumulative


In [ ]:
top_hansen = (
    coacc_hansen[["id", "name", "category", "coacc_population", "coacc_age_0_15", "coacc_age_65_plus"]]
    .sort_values("coacc_population", ascending=False)
    .head(15)
)
top_hansen


## Plot One Category on the Map


In [ ]:
pharmacy_cumulative = coacc_cumulative[coacc_cumulative["category"] == "amenity_pharmacy"].copy()
pharmacy_hansen = coacc_hansen[coacc_hansen["category"] == "amenity_pharmacy"].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
hex_ams.plot(ax=axes[0], color="whitesmoke", edgecolor="lightgray", linewidth=0.2)
pharmacy_cumulative.plot(
    ax=axes[0],
    column="coacc_population",
    cmap="YlOrRd",
    legend=True,
    markersize=10,
)
axes[0].set_title("Amsterdam pharmacies | cumulative co-accessibility")
axes[0].axis("off")

hex_ams.plot(ax=axes[1], color="whitesmoke", edgecolor="lightgray", linewidth=0.2)
pharmacy_hansen.plot(
    ax=axes[1],
    column="coacc_population",
    cmap="YlGnBu",
    legend=True,
    markersize=10,
)
axes[1].set_title("Amsterdam pharmacies | Hansen co-accessibility")
axes[1].axis("off")

plt.tight_layout()
plt.show()


## Compare the Population Groups for One POI Category


In [ ]:
pharmacy_group_summary = coacc_cumulative[coacc_cumulative["category"] == "amenity_pharmacy"][[
    "coacc_population",
    "coacc_men",
    "coacc_women",
    "coacc_age_0_15",
    "coacc_age_65_plus",
]].describe()
pharmacy_group_summary
